# 07. Dijkstra: kürzeste Wege mit Gewichten

In [Notebook 05](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/05_breitensuche.ipynb) hat die Breitensuche kürzeste Wege gefunden, solange **jeder Schritt
gleich viel kostet**. Oft stimmt das nicht: Eine Landstraße dauert länger als ein Stadtring, ein Umstieg länger als eine
Station, und in der Zauberschule dauert der Wechsel des Stockwerks dreimal so lange wie ein Schritt. Jede Kante bekommt dann ein
**Gewicht** (Minuten, Sekunden, Kosten), und gesucht ist der Weg mit dem **kleinsten Gesamtgewicht**. Dafür gibt es den
**Algorithmus von Dijkstra** (sprich *Deikstra*). Er gehört zu den wichtigsten Werkzeugen der 1. Runde.

In diesem Notebook lernst du:

- einen **gewichteten Graphen** als Wörterbuch von Listen aus Paaren `(nachbar, gewicht)` zu speichern,
- warum die Breitensuche hier **falsche** Wege liefert,
- die **Prioritätswarteschlange** `heapq` und Tupel `(entfernung, knoten)`,
- Dijkstra mit dem Überspringen **veralteter Einträge**, und warum ein Knoten beim Herausnehmen **endgültig** ist,
- den **Weg** zurückzuverfolgen,
- eine Alternative für kleine ganze Gewichte: **Zwischenknoten** und Breitensuche,
- Zustände mit Kosten: das **Umsteigen** im U-Bahn-Netz aus Notebook 05.

In der Wettbewerbs-Ecke hilfst du Ron jetzt wirklich durch beide Stockwerke der Zauberschule. Für Schnelle gibt es eine kleine
Version des ägyptischen Grabmals.

## 1. Eine Straßenkarte mit Fahrzeiten

Acht Orte sind durch zwölf Straßen verbunden. Die Zahl ist die Fahrzeit in Minuten, in beide Richtungen gleich:

| Straße | Minuten | Straße | Minuten | Straße | Minuten |
|---|---|---|---|---|---|
| Altdorf – Bergen | 4 | Altdorf – Cella | 2 | Cella – Bergen | 1 |
| Bergen – Dorn | 5 | Altdorf – Furth | 30 | Dorn – Furth | 3 |
| Cella – Eich | 10 | Eich – Furth | 7 | Furth – Gera | 6 |
| Dorn – Gera | 12 | Eich – Hohn | 4 | Hohn – Gera | 5 |

Wir speichern die Karte wie in Notebook 05 als Wörterbuch, aber zu jedem Ort gehört jetzt eine Liste von **Paaren**
`(nachbar, minuten)`. Das Paar ist ein Tupel, das man in `for nachbar, minuten in karte[ort]:` gleich auspacken kann.

#### Aufgabe 1.1

Schreibe `baue_karte(strassen)`, die aus der Liste der Straßen `(ort1, ort2, minuten)` das Wörterbuch macht. Jede Straße steht
zweimal darin, einmal bei jedem Ende.

**Erwartete Ausgabe:**
```
8 Orte
[('Bergen', 4), ('Cella', 2), ('Furth', 30)]
[('Furth', 6), ('Dorn', 12), ('Hohn', 5)]
```

In [ ]:
strassen = [('Altdorf', 'Bergen', 4), ('Altdorf', 'Cella', 2), ('Cella', 'Bergen', 1), ('Bergen', 'Dorn', 5),
            ('Altdorf', 'Furth', 30), ('Dorn', 'Furth', 3), ('Cella', 'Eich', 10), ('Eich', 'Furth', 7),
            ('Furth', 'Gera', 6), ('Dorn', 'Gera', 12), ('Eich', 'Hohn', 4), ('Hohn', 'Gera', 5)]

def baue_karte(strassen):
    karte = {}
    # dein Code
    return karte

karte = baue_karte(strassen)
print(len(karte), 'Orte')
if len(karte) > 0:
    print(karte['Altdorf'])
    print(karte['Gera'])

<details>
<summary>Tipp anzeigen</summary>

Erst jeden Ort mit einer leeren Liste eintragen, wenn er noch nicht im Wörterbuch steht. Dann bei `a` das Paar `(b, minuten)`
anhängen und bei `b` das Paar `(a, minuten)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
strassen = [('Altdorf', 'Bergen', 4), ('Altdorf', 'Cella', 2), ('Cella', 'Bergen', 1), ('Bergen', 'Dorn', 5),
            ('Altdorf', 'Furth', 30), ('Dorn', 'Furth', 3), ('Cella', 'Eich', 10), ('Eich', 'Furth', 7),
            ('Furth', 'Gera', 6), ('Dorn', 'Gera', 12), ('Eich', 'Hohn', 4), ('Hohn', 'Gera', 5)]

def baue_karte(strassen):
    karte = {}
    for a, b, minuten in strassen:
        if a not in karte:
            karte[a] = []
        if b not in karte:
            karte[b] = []
        karte[a].append((b, minuten))        # beide Richtungen eintragen
        karte[b].append((a, minuten))
    return karte

karte = baue_karte(strassen)
print(len(karte), 'Orte')
if len(karte) > 0:
    print(karte['Altdorf'])
    print(karte['Gera'])
```

</details>

Was macht die **Breitensuche** aus Notebook 05 auf dieser Karte? Hier ist sie, so umgeschrieben, dass sie mit den Paaren
zurechtkommt (die Minuten ignoriert sie einfach). `fahrzeit` addiert die Minuten entlang eines Weges.

In [ ]:
from collections import deque

def bfs_weg(karte, start, ziel):
    vorgaenger = {start: None}
    schlange = deque([start])
    while len(schlange) > 0 and ziel not in vorgaenger:
        v = schlange.popleft()
        for u, minuten in karte[v]:              # die Minuten werden nicht beachtet
            if u not in vorgaenger:
                vorgaenger[u] = v
                schlange.append(u)
    weg = []
    knoten = ziel
    while knoten is not None:
        weg.append(knoten)
        knoten = vorgaenger[knoten]
    weg.reverse()
    return weg

def fahrzeit(karte, weg):
    summe = 0
    for k in range(len(weg) - 1):
        for u, minuten in karte[weg[k]]:
            if u == weg[k + 1]:
                summe = summe + minuten
    return summe

weg_bfs = bfs_weg(karte, 'Altdorf', 'Furth')
print(weg_bfs, fahrzeit(karte, weg_bfs), 'Minuten')

#### Frage 1.2

Die Breitensuche findet den Weg mit den **wenigsten Straßen**: von Altdorf direkt nach Furth. Das dauert 30 Minuten. Finde
von Hand einen schnelleren Weg. Warum findet die Breitensuche ihn nicht?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Altdorf – Cella – Bergen – Dorn – Furth dauert 2 + 1 + 5 + 3 = **11 Minuten**, obwohl es vier Straßen sind. Die Breitensuche
zählt nur die **Anzahl** der Straßen, nicht ihre Länge. Sie hält den Abstand für richtig, den ein Ort beim Entdecken bekommt,
und das stimmt nur, wenn jeder Schritt gleich viel kostet. Hier kann ein Ort, der „weit“ entfernt ist (viele Straßen), näher
sein als einer mit direkter Straße. Wir brauchen ein Verfahren, das die Orte nach **Gesamtzeit** abarbeitet und nicht nach
Zahl der Straßen.

</details>

## 2. Dijkstra

Die Idee ist die der Breitensuche, aber die Warteschlange ist jetzt eine **Prioritätswarteschlange**: Herausgenommen wird immer
der Eintrag mit der **kleinsten Entfernung** vom Start. Man stellt sich vor, ein Fahrzeug fährt von allen Orten gleichzeitig
los. Der Ort, den es als Nächstes erreicht, hat den nächstkleineren Abstand.

**Prioritätswarteschlange mit `heapq`.** Eine normale Liste müsste man für jede Entnahme nach dem kleinsten Element
durchsuchen. Das Modul `heapq` ordnet eine Liste so an („Halde“, englisch *heap*), dass das Kleinste immer vorne steht, und
beide Operationen sind schnell (Zeit proportional zu `log n` für `n` Einträge):

- `heappush(liste, eintrag)` fügt einen Eintrag ein,
- `heappop(liste)` entfernt den **kleinsten** Eintrag und gibt ihn zurück.

Einträge sind **Tupel** `(entfernung, ort)`. Tupel werden **elementweise** verglichen: erst die Entfernung, bei gleicher
Entfernung der Ort. So steht der Ort mit der kleinsten Entfernung vorne.

#### Aufgabe 2.1: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
from heapq import heappush, heappop

halde = []
heappush(halde, (7, 'Dom'))
heappush(halde, (3, 'Zoo'))
heappush(halde, (5, 'Uni'))
heappush(halde, (3, 'Alt'))
print(heappop(halde))
print(heappop(halde))
print(len(halde))
print(halde[0])

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
(3, 'Alt')
(3, 'Zoo')
2
(5, 'Uni')
```

Die beiden Einträge mit Entfernung 3 kommen zuerst. Weil die Entfernungen gleich sind, entscheidet das zweite Element des
Tupels: `'Alt'` ist (alphabetisch) kleiner als `'Zoo'`. Danach bleiben zwei Einträge übrig. `halde[0]` ist immer der kleinste
Eintrag, man sieht ihn sich an, ohne ihn herauszunehmen. Den Rest der Liste solltest du nicht ausgeben oder verändern: Die
Reihenfolge der übrigen Einträge ist nur „halb“ sortiert.

</details>

**Der Ablauf von Dijkstra.**

1. `entfernung = {start: 0}` und die Warteschlange enthält `(0, start)`.
2. Solange die Warteschlange nicht leer ist: den kleinsten Eintrag `(d, v)` herausnehmen.
3. Ist `d > entfernung[v]`, ist der Eintrag **veraltet** (für `v` wurde inzwischen ein kürzerer Weg gefunden): überspringen.
4. Sonst für jeden Nachbarn `(u, minuten)` von `v`: `neu = d + minuten`. Ist `u` neu oder ist `neu < entfernung[u]`, trage
   `entfernung[u] = neu` ein und stelle `(neu, u)` in die Warteschlange.

Warum gibt es veraltete Einträge? `heapq` kann einen Eintrag nicht ändern. Findet man später einen **kürzeren** Weg zu `u`,
legt man einfach einen neuen Eintrag an. Der alte bleibt in der Halde liegen, kommt aber später heraus als `(d, u)` mit
`d > entfernung[u]` und wird übersprungen.

#### Aufgabe 2.2

Schreibe `entfernungen(karte, start)`. Sie gibt das Wörterbuch der kürzesten Fahrzeiten vom Start zu allen erreichbaren Orten
zurück, dazu die Zahl der **herausgenommenen** Einträge und wie viele davon **veraltet** waren.

**Erwartete Ausgabe:**
```
{'Altdorf': 0, 'Bergen': 3, 'Cella': 2, 'Furth': 11, 'Eich': 12, 'Dorn': 8, 'Gera': 17, 'Hohn': 16}
11 Einträge herausgenommen, davon 3 veraltet
```

In [ ]:
from heapq import heappush, heappop

def entfernungen(karte, start):
    entfernung = {start: 0}
    warteschlange = [(0, start)]
    herausgenommen = 0
    veraltet = 0
    # dein Code
    return entfernung, herausgenommen, veraltet

ent_altdorf, herausgenommen, veraltet = entfernungen(karte, 'Altdorf')
print(ent_altdorf)
print(herausgenommen, 'Einträge herausgenommen, davon', veraltet, 'veraltet')

<details>
<summary>Tipp anzeigen</summary>

Die Schleife `while len(warteschlange) > 0:`. Darin `d, v = heappop(warteschlange)`, dann prüfen, ob `d > entfernung[v]` ist
(dann `continue`), dann die Nachbarn mit `for u, minuten in karte[v]:`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
from heapq import heappush, heappop

def entfernungen(karte, start):
    entfernung = {start: 0}
    warteschlange = [(0, start)]
    herausgenommen = 0
    veraltet = 0
    while len(warteschlange) > 0:
        d, v = heappop(warteschlange)            # kleinste Entfernung zuerst
        herausgenommen = herausgenommen + 1
        if d > entfernung[v]:                    # veralteter Eintrag
            veraltet = veraltet + 1
            continue
        for u, minuten in karte[v]:
            neu = d + minuten
            if u not in entfernung or neu < entfernung[u]:
                entfernung[u] = neu              # besser als alles bisher
                heappush(warteschlange, (neu, u))
    return entfernung, herausgenommen, veraltet

ent_altdorf, herausgenommen, veraltet = entfernungen(karte, 'Altdorf')
print(ent_altdorf)
print(herausgenommen, 'Einträge herausgenommen, davon', veraltet, 'veraltet')
```

Furth bekommt zuerst den Wert 30 (direkte Straße), später 11 über Dorn. Auch Bergen (erst 4, dann 3 über Cella) und Gera (erst 20
über Dorn, dann 17 über Furth) werden zweimal eingetragen. Dadurch gibt es drei veraltete Einträge, die beim Herausnehmen
übersprungen werden. Die Orte stehen im Wörterbuch in der
Reihenfolge ihrer **ersten** Entdeckung, die Werte sind die **endgültigen**.

</details>

#### Frage 2.3

1. Warum ist die Entfernung eines Ortes **endgültig**, wenn sein Eintrag zum ersten Mal (nicht veraltet) herauskommt?
2. Was geht schief, wenn eine Straße ein **negatives** Gewicht hätte?
3. Wie groß ist der Aufwand für `V` Orte und `E` Straßen?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Der Eintrag `(d, v)` ist der **kleinste** in der Warteschlange. Jeder andere Weg zu `v` müsste über einen Ort führen, der
   noch nicht herausgenommen wurde. Der steht mit mindestens `d` in der Warteschlange (oder wird erst später eingetragen mit
   noch mehr), und danach kommen nur noch **Gewichte, die nicht negativ sind**. Der Weg wäre also mindestens `d` lang. Es gibt
   keinen kürzeren Weg, und `d` ist endgültig.
2. Genau diese Begründung bricht zusammen. Beispiel: Von `A` führt eine Straße mit 2 nach `B` und eine mit 3 nach `C`, von `C`
   nach `B` eine mit −2. Dijkstra nimmt `B` mit 2 heraus und hielte es für fertig, aber über `C` ist es nur 3 − 2 = 1. Mit
   negativen Gewichten braucht man andere Verfahren (zum Beispiel Bellman-Ford). In den Aufgaben der 1. Runde kommt das kaum
   vor, Zeiten und Längen sind nicht negativ.
3. Jede Straße führt höchstens einmal zu einem neuen Eintrag, es gibt also höchstens `E` Einträge (plus den Start). Jeder
   Eintrag wird einmal eingefügt und einmal herausgenommen, das kostet je `O(log V)` (in der Halde ist `log E` höchstens
   `2 · log V`). Zusammen **O((V + E) · log V)**. Das ist nur um den Faktor `log V` langsamer als die Breitensuche. Sind alle
   Gewichte 1, ist Dijkstra dasselbe Verfahren wie die Breitensuche, aber mit der langsameren Warteschlange.

</details>

## 3. Den Weg zurückverfolgen

Wie in Notebook 05 merkt man sich zu jedem Ort den **Vorgänger**, über den die beste Entfernung gefunden wurde. Wird die
Entfernung später verbessert, wird auch der Vorgänger überschrieben. Weil ein Ort **endgültig** ist, sobald er herauskommt,
kann man aufhören, sobald das **Ziel** herauskommt.

#### Aufgabe 3.1

Schreibe `schnellster_weg(graph, start, ziel)`. Sie gibt ein Paar `(zeit, weg)` mit der Liste der Orte zurück, oder `None`, wenn
das Ziel nicht erreichbar ist (oder gar nicht vorkommt).

**Erwartete Ausgabe:**
```
Furth (11, ['Altdorf', 'Cella', 'Bergen', 'Dorn', 'Furth'])
Gera (17, ['Altdorf', 'Cella', 'Bergen', 'Dorn', 'Furth', 'Gera'])
Isenburg None
```

In [ ]:
def schnellster_weg(graph, start, ziel):
    entfernung = {start: 0}
    vorgaenger = {start: None}
    warteschlange = [(0, start)]
    # dein Code
    return None

for ort in ['Furth', 'Gera', 'Isenburg']:
    print(ort, schnellster_weg(karte, 'Altdorf', ort))

<details>
<summary>Tipp anzeigen</summary>

Wie in 2.2, dazu `vorgaenger[u] = v` immer dann, wenn `entfernung[u]` gesetzt wird. Kommt `v == ziel` aus der Warteschlange, geh
mit `v = vorgaenger[v]` zurück bis `None` und dreh die Liste um. Läuft die Warteschlange leer, gib `None` zurück.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def schnellster_weg(graph, start, ziel):
    entfernung = {start: 0}
    vorgaenger = {start: None}
    warteschlange = [(0, start)]
    while len(warteschlange) > 0:
        d, v = heappop(warteschlange)
        if d > entfernung[v]:
            continue
        if v == ziel:                            # endgültig: fertig
            weg = []
            while v is not None:
                weg.append(v)
                v = vorgaenger[v]
            weg.reverse()
            return d, weg
        for u, kosten in graph[v]:
            neu = d + kosten
            if u not in entfernung or neu < entfernung[u]:
                entfernung[u] = neu
                vorgaenger[u] = v                # auch der Vorgänger wird überschrieben
                heappush(warteschlange, (neu, u))
    return None

for ort in ['Furth', 'Gera', 'Isenburg']:
    print(ort, schnellster_weg(karte, 'Altdorf', ort))
```

Der Weg nach Furth hat vier Straßen und dauert 11 Minuten, die Breitensuche hatte die Direktstraße mit 30 Minuten gewählt.
Für `'Isenburg'`, das es auf der Karte nicht gibt, läuft die Warteschlange leer, und die Funktion gibt `None` zurück.

</details>

## 4. Zwei typische Fehler

Beide Fehler sind kleine Änderungen an `entfernungen` aus 2.2.

#### Frage 4.1

Finde den Fehler. Tim übernimmt die Idee der Breitensuche: Ein Ort bekommt seine Entfernung, **sobald er entdeckt** wird,
und wird danach nie mehr geändert.

```python
def entfernungen_falsch(karte, start):
    entfernung = {start: 0}
    warteschlange = [(0, start)]
    while len(warteschlange) > 0:
        d, v = heappop(warteschlange)
        for u, minuten in karte[v]:
            if u not in entfernung:              # wie bei der Breitensuche
                entfernung[u] = d + minuten
                heappush(warteschlange, (d + minuten, u))
    return entfernung
```

Für Altdorf liefert das 4 Minuten nach Bergen (richtig: 3), 9 nach Dorn (richtig: 8) und
30 nach Furth (richtig: 11). Warum?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Von Altdorf aus werden Bergen (4), Cella (2) und Furth (30) sofort entdeckt und bekommen ihren Wert. Später kommt Cella heraus
und findet Bergen mit 2 + 1 = 3, aber `u not in entfernung` ist falsch, also wird nichts geändert. Genauso bleibt Furth bei 30, auch
wenn Dorn einen Weg mit 11 findet.

Bei der Breitensuche ist der **erste** Weg zu einem Ort auch der kürzeste, weil alles nach Schrittzahl geordnet wird. Bei
Dijkstra ist ein beim Entdecken eingetragener Wert nur **vorläufig**. Er darf verbessert werden (`neu < entfernung[u]`) und ist
erst **endgültig**, wenn sein Eintrag herauskommt.

</details>

#### Frage 4.2

Anna vergisst das Überspringen der veralteten Einträge. In diesem Programm kann man es mit dem Parameter `ueberspringen`
ein- und ausschalten:

```python
def zaehle_kantenpruefungen(karte, start, ueberspringen):
    entfernung = {start: 0}
    warteschlange = [(0, start)]
    pruefungen = 0
    while len(warteschlange) > 0:
        d, v = heappop(warteschlange)
        if ueberspringen and d > entfernung[v]:
            continue
        for u, minuten in karte[v]:
            pruefungen = pruefungen + 1
            if u not in entfernung or d + minuten < entfernung[u]:
                entfernung[u] = d + minuten
                heappush(warteschlange, (d + minuten, u))
    return entfernung, pruefungen
```

Auf der Karte kommt in beiden Fällen dasselbe Wörterbuch heraus. Mit Überspringen werden aber 24 Straßen angesehen,
ohne sind es 34. Wie kann das sein, und wann wird der Unterschied zum Problem?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Das Ergebnis bleibt richtig: Ein veralteter Eintrag hat eine zu große Entfernung, deshalb schlägt `d + minuten < entfernung[u]`
bei seinen Nachbarn nie an, und es wird nichts verschlechtert. Aber die Schleife über alle Nachbarn läuft trotzdem, und das
sind unnötige Prüfungen (34 statt 24). Bei Graphen mit vielen Straßen je Ort und vielen Verbesserungen
(zum Beispiel die Felder der Zauberschule oder große Netze) wächst die Zahl der veralteten Einträge und damit die Rechenzeit
spürbar. Die eine Zeile `if d > entfernung[v]: continue` ist deshalb ein fester Bestandteil von Dijkstra.

</details>

## 5. Alternative: Zwischenknoten und Breitensuche

Sind alle Gewichte **kleine ganze Zahlen**, kann man ganz ohne `heapq` auskommen: Eine Straße mit 5 Minuten ersetzt man durch
eine Kette von 5 Straßen mit je 1 Minute, mit 4 neuen **Zwischenknoten**. Im neuen Graphen kostet jede Kante 1, und die
Breitensuche aus Notebook 05 findet die kürzesten Wege. Eine Entfernung von `a` nach `b` ist dasselbe wie vorher. Die Zwischenknoten
benennen wir als Tupel `(a, b, k)`: der `k`-te Zwischenknoten auf der Straße von `a` nach `b`. Für die Gegenrichtung gibt es eine
eigene Kette `(b, a, k)`.

#### Aufgabe 5.1

Schreibe `zerlege(karte)`, die aus der gewichteten Karte einen Graphen ohne Gewichte macht (Wörterbuch von Listen von Nachbarn).
Die Breitensuche `abstaende` ist fertig. Prüfe, dass die Entfernungen zu den echten Orten mit denen von Dijkstra
übereinstimmen.

**Erwartete Ausgabe:**
```
8 Orte, 162 Knoten nach dem Zerlegen
gleiche Entfernungen wie Dijkstra: True
```

In [ ]:
from collections import deque

def abstaende(graph, start):
    abstand = {start: 0}
    schlange = deque([start])
    while len(schlange) > 0:
        v = schlange.popleft()
        for u in graph[v]:
            if u not in abstand:
                abstand[u] = abstand[v] + 1
                schlange.append(u)
    return abstand

def zerlege(karte):
    graph = {}
    # dein Code
    return graph

einfach = zerlege(karte)
print(len(karte), 'Orte,', len(einfach), 'Knoten nach dem Zerlegen')
gleich = False
if len(einfach) > 0:
    abstand_altdorf = abstaende(einfach, 'Altdorf')
    gleich = True
    for ort in karte:
        if abstand_altdorf[ort] != ent_altdorf[ort]:
            gleich = False
print('gleiche Entfernungen wie Dijkstra:', gleich)

<details>
<summary>Tipp anzeigen</summary>

Zuerst alle Orte mit leerer Liste eintragen. Für jede Straße `(a, b, minuten)` eine Kette: `vorher = a`, dann `minuten - 1` Mal
einen neuen Knoten `(a, b, k)` anlegen, `vorher` mit ihm verbinden und `vorher` weitersetzen. Am Ende `vorher` mit `b` verbinden.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
from collections import deque

def abstaende(graph, start):
    abstand = {start: 0}
    schlange = deque([start])
    while len(schlange) > 0:
        v = schlange.popleft()
        for u in graph[v]:
            if u not in abstand:
                abstand[u] = abstand[v] + 1
                schlange.append(u)
    return abstand

def zerlege(karte):
    graph = {}
    for ort in karte:
        graph[ort] = []
    for a in karte:
        for b, minuten in karte[a]:
            vorher = a
            for k in range(1, minuten):          # minuten - 1 Zwischenknoten
                zwischen = (a, b, k)
                graph[zwischen] = []
                graph[vorher].append(zwischen)
                vorher = zwischen
            graph[vorher].append(b)              # letzte Kante der Kette führt zu b
    return graph

einfach = zerlege(karte)
print(len(karte), 'Orte,', len(einfach), 'Knoten nach dem Zerlegen')
gleich = False
if len(einfach) > 0:
    abstand_altdorf = abstaende(einfach, 'Altdorf')
    gleich = True
    for ort in karte:
        if abstand_altdorf[ort] != ent_altdorf[ort]:
            gleich = False
print('gleiche Entfernungen wie Dijkstra:', gleich)
```

</details>

#### Frage 5.2

Der zerlegte Graph hat 162 Knoten statt 8. Wann lohnt sich diese Methode, wann nicht?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Es lohnt sich**, wenn die Gewichte klein sind (wie die 3 Sekunden für den Stockwerkwechsel in der Zauberschule): Man spart
`heapq` und kommt mit einer Breitensuche aus, die ein wenig schneller ist (`O(V + E)` statt `O((V + E) · log V)`), und sie
ist leicht zu schreiben.

**Es lohnt sich nicht**, wenn die Gewichte groß sind oder keine ganzen Zahlen: Eine Straße mit 1 000 000 Minuten ergäbe eine Million
Zwischenknoten, und bei Kommazahlen geht es gar nicht. Dijkstra sieht jede Straße nur einmal, egal wie lang sie ist. Die Zahl der
neuen Knoten ist die Summe aller Gewichte minus 1 je Kante: auf dieser Karte 2 · 89 − 24 + 8 = 162 (89 ist die Summe aller
Fahrzeiten, jede Straße hat zwei Ketten, es gibt 24 Kanten und 8 Orte).

</details>

## 6. Umsteigen: Zustände mit Kosten

Am Ende von Notebook 05 stand die Frage: Der Weg Hafen – Messe mit den wenigsten Fahrten hat 4 Fahrten, aber 3 Umstiege. Der
Weg mit dem einzigen Umstieg hat 7 Fahrten. Welcher ist „besser“, hängt davon ab, **wie viel ein Umstieg im Vergleich zu einer
Fahrt kostet**. Wir sagen: Eine Fahrt zur nächsten Station kostet 1, ein Umstieg 5 (Laufen und Warten). Die Zahlen kann man
später ändern.

Dazu sind die Knoten **Zustände** (Notebook 06): das Paar `(station, linie)` heißt „ich sitze in dieser Linie an dieser Station“.
Die Kanten:

- Fahrt: `(station, linie)` nach `(nächste Station, linie)`, Kosten `fahrt`,
- Umstieg: `(station, linie1)` nach `(station, linie2)` für zwei Linien, die an der Station halten, Kosten `umstieg`.

Alle Kanten gelten in beide Richtungen. Das ist ein gewichteter Graph wie die Straßenkarte, und `schnellster_weg` aus 3.1 geht
unverändert.

#### Aufgabe 6.1

Schreibe `baue_umsteignetz(linien, fahrt, umstieg)`. Sie gibt das Wörterbuch des Graphen zurück. Die Hilfsfunktion `verbinde`,
die eine Kante in beide Richtungen einträgt, ist fertig.

**Erwartete Ausgabe:**
```
16 Zustände
['U1', 'U2']
[(('Hafen', 'U1'), 1), (('Dom', 'U1'), 1), (('Markt', 'U2'), 5)]
```

In [ ]:
linien = {'U1': ['Hafen', 'Markt', 'Dom', 'Uni', 'Zoo'],
          'U2': ['Bahnhof', 'Markt', 'Rathaus', 'Park'],
          'U3': ['Zoo', 'Park', 'Stadion', 'Messe'],
          'U4': ['Dom', 'Rathaus', 'Stadion']}

def verbinde(netz, a, b, kosten):
    if a not in netz:
        netz[a] = []
    if b not in netz:
        netz[b] = []
    netz[a].append((b, kosten))
    netz[b].append((a, kosten))

def baue_umsteignetz(linien, fahrt, umstieg):
    netz = {}
    halten = {}                      # halten[station] = Liste der Linien, die dort halten
    # dein Code
    return netz, halten

netz, halten = baue_umsteignetz(linien, 1, 5)
print(len(netz), 'Zustände')
if len(netz) > 0:
    print(halten['Markt'])
    print(netz[('Markt', 'U1')])

<details>
<summary>Tipp anzeigen</summary>

Zwei Teile. Erstens für jede Linie und jede Position `i`: die Linie bei `halten[station]` eintragen und, wenn es eine nächste
Station gibt, `verbinde` mit Kosten `fahrt` aufrufen. Zweitens für jede Station alle Paare `i < j` von Linien in
`halten[station]` mit Kosten `umstieg` verbinden.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
linien = {'U1': ['Hafen', 'Markt', 'Dom', 'Uni', 'Zoo'],
          'U2': ['Bahnhof', 'Markt', 'Rathaus', 'Park'],
          'U3': ['Zoo', 'Park', 'Stadion', 'Messe'],
          'U4': ['Dom', 'Rathaus', 'Stadion']}

def verbinde(netz, a, b, kosten):
    if a not in netz:
        netz[a] = []
    if b not in netz:
        netz[b] = []
    netz[a].append((b, kosten))
    netz[b].append((a, kosten))

def baue_umsteignetz(linien, fahrt, umstieg):
    netz = {}
    halten = {}                      # halten[station] = Liste der Linien, die dort halten
    for name in linien:
        stationen = linien[name]
        for i in range(len(stationen)):
            if stationen[i] not in halten:
                halten[stationen[i]] = []
            halten[stationen[i]].append(name)
            if i + 1 < len(stationen):                  # Fahrt zur nächsten Station
                verbinde(netz, (stationen[i], name), (stationen[i + 1], name), fahrt)
    for station in halten:
        for i in range(len(halten[station])):           # jedes Paar von Linien an dieser Station
            for j in range(i + 1, len(halten[station])):
                verbinde(netz, (station, halten[station][i]), (station, halten[station][j]), umstieg)
    return netz, halten

netz, halten = baue_umsteignetz(linien, 1, 5)
print(len(netz), 'Zustände')
if len(netz) > 0:
    print(halten['Markt'])
    print(netz[('Markt', 'U1')])
```

</details>

#### Aufgabe 6.2

Schreibe `fahre(linien, start, ziel, fahrt, umstieg)`. Der Start ist keine einzelne Station mit Linie: Man kann an `start`
in **jede** Linie einsteigen, ohne zu bezahlen. Lege dazu einen zusätzlichen Knoten `ANFANG` an, der mit Kosten 0 mit allen
`(start, linie)` verbunden ist, und einen Knoten `ENDE` mit Kosten 0 zu allen `(ziel, linie)`. Dann ist es ein Aufruf von
`schnellster_weg(netz, ANFANG, ENDE)`. `ANFANG` und `ENDE` sind Tupel wie die anderen Knoten, denn die Halde muss Einträge
`(entfernung, knoten)` vergleichen können, und ein Text lässt sich nicht mit einem Tupel vergleichen. Gib `(kosten, fahrten, umstiege)` zurück, die Fahrten und Umstiege zählst du auf dem
Weg (ohne `ANFANG` und `ENDE`) ab: Bleibt die Linie gleich, ist es eine Fahrt, sonst ein Umstieg.

**Erwartete Ausgabe:**
```
Umstieg kostet 0 : 4 Minuten, 4 Fahrten, 3 Umstiege
Umstieg kostet 1 : 7 Minuten, 5 Fahrten, 2 Umstiege
Umstieg kostet 5 : 12 Minuten, 7 Fahrten, 1 Umstiege
Umstieg kostet 20 : 27 Minuten, 7 Fahrten, 1 Umstiege
```

In [ ]:
ANFANG = ('Start', '')                           # Tupel, damit sie mit (station, linie) vergleichbar sind
ENDE = ('Ziel', '')

def fahre(linien, start, ziel, fahrt, umstieg):
    netz, halten = baue_umsteignetz(linien, fahrt, umstieg)
    # dein Code
    return None

for umstieg in [0, 1, 5, 20]:
    ergebnis = fahre(linien, 'Hafen', 'Messe', 1, umstieg)
    if ergebnis is not None:
        print('Umstieg kostet', umstieg, ':', ergebnis[0], 'Minuten,', ergebnis[1], 'Fahrten,', ergebnis[2], 'Umstiege')

<details>
<summary>Tipp anzeigen</summary>

`verbinde(netz, ANFANG, (start, name), 0)` für jede Linie `name` in `halten[start]`, ebenso für das Ziel. Auf dem Weg
`weg[0]` ist `ANFANG`, `weg[-1]` ist `ENDE`. Dazwischen vergleichst du bei aufeinanderfolgenden Zuständen die Linie
(`weg[k][1]`).

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
ANFANG = ('Start', '')                           # Tupel, damit sie mit (station, linie) vergleichbar sind
ENDE = ('Ziel', '')

def fahre(linien, start, ziel, fahrt, umstieg):
    netz, halten = baue_umsteignetz(linien, fahrt, umstieg)
    for name in halten[start]:
        verbinde(netz, ANFANG, (start, name), 0)
    for name in halten[ziel]:
        verbinde(netz, (ziel, name), ENDE, 0)
    kosten, weg = schnellster_weg(netz, ANFANG, ENDE)
    fahrten = 0
    umstiege = 0
    for k in range(1, len(weg) - 2):             # nur die Zustände (station, linie)
        if weg[k][1] == weg[k + 1][1]:
            fahrten = fahrten + 1
        else:
            umstiege = umstiege + 1
    return kosten, fahrten, umstiege

for umstieg in [0, 1, 5, 20]:
    ergebnis = fahre(linien, 'Hafen', 'Messe', 1, umstieg)
    if ergebnis is not None:
        print('Umstieg kostet', umstieg, ':', ergebnis[0], 'Minuten,', ergebnis[1], 'Fahrten,', ergebnis[2], 'Umstiege')
```

</details>

#### Frage 6.3

Die Ausgabe zeigt: Kostet ein Umstieg 0, nimmt man den Weg mit den wenigsten Fahrten (4 Fahrten, 3 Umstiege). Bei 1 sind es
5 Fahrten und 2 Umstiege, ab 5 der Weg mit nur einem Umstieg und 7 Fahrten.
1. Rechne die Kosten der drei Wege als Formel in `u` (Kosten eines Umstiegs) aus: 4 Fahrten und 3 Umstiege, 5 Fahrten und 2
   Umstiege, 7 Fahrten und 1 Umstieg.
2. Ab welchem `u` ist der Weg mit einem Umstieg **eindeutig** der beste?
3. Warum konnte Notebook 05 die Frage nicht beantworten?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. 4 + 3 · u, 5 + 2 · u und 7 + u.
2. 7 + u < 5 + 2 · u gilt für u > 2, und 7 + u < 4 + 3 · u gilt für u > 1,5. Ab u = 3 ist der Weg mit einem Umstieg der beste. Bei
   u = 2 sind der Weg mit zwei Umstiegen und der mit einem gleich gut (je 9), und bei u = 1 sind es der mit drei und der mit
   zwei Umstiegen (je 7). Dijkstra liefert dann einen von beiden, deshalb sieht man bei 1 nur den mit 5 Fahrten.
3. Die Breitensuche zählt jede Kante gleich. Hier kosten Fahrt und Umstieg **verschieden viel**, und mit Zwischenknoten ginge es
   nur, wenn `u` eine kleine ganze Zahl ist. Dijkstra kann beliebige Kosten, auch 2,5 für einen Umstieg. Außerdem war ein
   Zustand `(station, linie)` statt nur `station` nötig, denn ob man umsteigt, hängt davon ab, in welcher Linie man sitzt. Das
   ist die Idee aus Notebook 06.

</details>

## Wettbewerbs-Ecke 1: Zauberschule mit beiden Stockwerken

*Angelehnt an Aufgabe 3 „Zauberschule“ der 1. Runde des 42. Bundeswettbewerbs Informatik (2023/24).*

Die Zauberschule Bugwarts hat zwei Stockwerke mit Feldern und Wänden. Ron will so schnell wie möglich von Feld `A` nach Feld
`B`. Ein Schritt zum Nachbarfeld dauert 1 Sekunde. Ein Wechsel ins andere Stockwerk, auf das Feld **genau darüber oder darunter**,
dauert 3 Sekunden, und das Zielfeld im anderen Stockwerk darf keine Wand sein. In Notebook 05 durfte Ron nur im oberen
Stockwerk bleiben. Jetzt darf er beide benutzen.

Ein Knoten ist ein Zustand `(stockwerk, zeile, spalte)` mit `stockwerk` 0 (oben) oder 1 (unten). Dijkstra ist genau das, was
wir brauchen: Die Kanten kosten 1 oder 3.

#### Aufgabe E.1

Schreibe `nachbarn_schule(plan, zustand)`. Sie gibt die Liste aller Paare `(nachbarzustand, kosten)` zurück: bis zu vier Schritte
im selben Stockwerk (Kosten 1, kein `#`, im Plan) und den Wechsel ins andere Stockwerk (Kosten 3, dort keine Wand).
`plan` ist die Liste der beiden Stockwerke, jedes eine Liste von Textzeilen. `lies_plan` ist fertig.

**Erwartete Ausgabe:**
```
(0, 9, 5) (0, 9, 7)
[((0, 9, 4), 1), ((0, 8, 5), 1), ((1, 9, 5), 3)]
```

In [ ]:
RICHTUNGEN = [(0, 1), (0, -1), (1, 0), (-1, 0)]          # rechts, links, unten, oben

def lies_plan(dateiname):
    zeilen = open(dateiname).read().split('\n')
    n = int(zeilen[0].split()[0])
    return [zeilen[1:1 + n], zeilen[2 + n:2 + 2 * n]]

def finde(plan, zeichen):
    for r in range(len(plan[0])):
        for c in range(len(plan[0][r])):
            if plan[0][r][c] == zeichen:
                return (0, r, c)
    return None

def nachbarn_schule(plan, zustand):
    stock, r, c = zustand
    liste = []
    # dein Code
    return liste

plan0 = lies_plan('daten/zauberschule0.txt')
start0 = finde(plan0, 'A')
ziel0 = finde(plan0, 'B')
print(start0, ziel0)
print(nachbarn_schule(plan0, start0))

<details>
<summary>Tipp anzeigen</summary>

Wie `nachbarfelder` in Notebook 05, nur mit `plan[stock]` statt `raster` und mit dem Kostenwert 1 im Paar. Der Wechsel:
`1 - stock` ist das andere Stockwerk, an derselben Stelle `(r, c)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
RICHTUNGEN = [(0, 1), (0, -1), (1, 0), (-1, 0)]          # rechts, links, unten, oben

def lies_plan(dateiname):
    zeilen = open(dateiname).read().split('\n')
    n = int(zeilen[0].split()[0])
    return [zeilen[1:1 + n], zeilen[2 + n:2 + 2 * n]]

def finde(plan, zeichen):
    for r in range(len(plan[0])):
        for c in range(len(plan[0][r])):
            if plan[0][r][c] == zeichen:
                return (0, r, c)
    return None

def nachbarn_schule(plan, zustand):
    stock, r, c = zustand
    liste = []
    for dz, ds in RICHTUNGEN:
        r2 = r + dz
        c2 = c + ds
        if 0 <= r2 < len(plan[stock]) and 0 <= c2 < len(plan[stock][r2]) and plan[stock][r2][c2] != '#':
            liste.append(((stock, r2, c2), 1))
    if plan[1 - stock][r][c] != '#':                     # Wechsel ins andere Stockwerk
        liste.append(((1 - stock, r, c), 3))
    return liste

plan0 = lies_plan('daten/zauberschule0.txt')
start0 = finde(plan0, 'A')
ziel0 = finde(plan0, 'B')
print(start0, ziel0)
print(nachbarn_schule(plan0, start0))
```

</details>

#### Aufgabe E.2

Schreibe `schnellste_route(plan, start, ziel)` mit Dijkstra und `nachbarn_schule`. Sie gibt `(sekunden, weg, herausgenommen)`
zurück: die Dauer, die Liste der Zustände und die Zahl der aus der Halde herausgenommenen Einträge.

**Erwartete Ausgabe:**
```
8 Sekunden, 4 Schritte und Wechsel, 37 Einträge herausgenommen
```

In [ ]:
def schnellste_route(plan, start, ziel):
    entfernung = {start: 0}
    vorgaenger = {start: None}
    warteschlange = [(0, start)]
    herausgenommen = 0
    # dein Code
    return None

ergebnis0 = schnellste_route(plan0, start0, ziel0)
if ergebnis0 is not None:
    sekunden0, weg0, herausgenommen0 = ergebnis0
    print(sekunden0, 'Sekunden,', len(weg0) - 1, 'Schritte und Wechsel,', herausgenommen0, 'Einträge herausgenommen')

<details>
<summary>Tipp anzeigen</summary>

Genau `schnellster_weg` aus 3.1, nur mit `for u, kosten in nachbarn_schule(plan, v):` und mit dem Zähler `herausgenommen`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def schnellste_route(plan, start, ziel):
    entfernung = {start: 0}
    vorgaenger = {start: None}
    warteschlange = [(0, start)]
    herausgenommen = 0
    while len(warteschlange) > 0:
        d, v = heappop(warteschlange)
        herausgenommen = herausgenommen + 1
        if d > entfernung[v]:
            continue
        if v == ziel:
            weg = []
            while v is not None:
                weg.append(v)
                v = vorgaenger[v]
            weg.reverse()
            return d, weg, herausgenommen
        for u, kosten in nachbarn_schule(plan, v):
            neu = d + kosten
            if u not in entfernung or neu < entfernung[u]:
                entfernung[u] = neu
                vorgaenger[u] = v
                heappush(warteschlange, (neu, u))
    return None

ergebnis0 = schnellste_route(plan0, start0, ziel0)
if ergebnis0 is not None:
    sekunden0, weg0, herausgenommen0 = ergebnis0
    print(sekunden0, 'Sekunden,', len(weg0) - 1, 'Schritte und Wechsel,', herausgenommen0, 'Einträge herausgenommen')
```

In `zauberschule0` braucht Ron statt 26 Sekunden im oberen Stockwerk jetzt nur 8: Er wechselt nach unten, geht ein Stück und
wechselt wieder nach oben.

</details>

#### Aufgabe E.3

Schreibe `zeichne_schule(plan, weg)`. Sie gibt beide Stockwerke mit dem Weg aus: `>` rechts, `<` links, `v` hinunter, `^` hinauf
(Zeilen zählen nach unten) und `!` für den Stockwerkwechsel. Das Zeichen steht auf dem Feld, **von dem aus** Ron weitergeht. `A`
und `B` bleiben stehen.

**Erwartete Ausgabe:**
```
#############
#.....#.....#
#.###.#.###.#
#...#.#...#.#
###.#.###.#.#
#...#.....#.#
#.#########.#
#.....#.....#
#####.#.###.#
#....A#B..#.#
#.#########.#
#...........#
#############

#############
#.......#...#
#...#.#.#...#
#...#.#.....#
#.###.#.#.###
#.....#.#...#
#####.###...#
#.....#.....#
#.#########.#
#...#>>!....#
#.#.#.#.###.#
#.#...#...#.#
#############
```

In [ ]:
def zeichne_schule(plan, weg):
    stockwerke = []
    for stock in plan:
        zeilen = []
        for zeile in stock:
            zeilen.append(list(zeile))
        stockwerke.append(zeilen)
    # dein Code
    for zeilen in stockwerke:
        for zeile in zeilen:
            print(''.join(zeile))
        print()

zeichne_schule(plan0, weg0)

<details>
<summary>Tipp anzeigen</summary>

Für jedes `k` bis `len(weg) - 2` die Zustände `weg[k]` und `weg[k + 1]` vergleichen. Haben sie ein unterschiedliches Stockwerk,
ist es `!`, sonst entscheiden die Unterschiede in Zeile und Spalte. Ins Stockwerk `stockwerke[stock]` schreiben, in dem Ron
gerade steht.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def zeichne_schule(plan, weg):
    stockwerke = []
    for stock in plan:
        zeilen = []
        for zeile in stock:
            zeilen.append(list(zeile))
        stockwerke.append(zeilen)
    for k in range(len(weg) - 1):
        stock, r, c = weg[k]
        stock2, r2, c2 = weg[k + 1]
        if stock2 != stock:
            zeichen = '!'
        elif c2 == c + 1:
            zeichen = '>'
        elif c2 == c - 1:
            zeichen = '<'
        elif r2 == r + 1:
            zeichen = 'v'
        else:
            zeichen = '^'
        if stockwerke[stock][r][c] not in 'AB':          # A bleibt stehen
            stockwerke[stock][r][c] = zeichen
    for zeilen in stockwerke:
        for zeile in zeilen:
            print(''.join(zeile))
        print()

zeichne_schule(plan0, weg0)
```

</details>

#### Aufgabe E.4

Bestimme für alle sechs Beispiele die Dauer und die Zahl der herausgenommenen Einträge. Vergleiche mit den Werten aus Notebook 05
(nur oberes Stockwerk: 26, 4, 32, 28, 674 und 1030 Sekunden).

**Erwartete Ausgabe:**
```
zauberschule0 : 8 Sekunden, 37 Einträge herausgenommen, bei 338 Feldern (mit Wänden)
zauberschule1 : 4 Sekunden, 6 Einträge herausgenommen, bei 462 Feldern (mit Wänden)
zauberschule2 : 14 Sekunden, 67 Einträge herausgenommen, bei 1350 Feldern (mit Wänden)
zauberschule3 : 28 Sekunden, 228 Einträge herausgenommen, bei 1922 Feldern (mit Wänden)
zauberschule4 : 84 Sekunden, 5045 Einträge herausgenommen, bei 20402 Feldern (mit Wänden)
zauberschule5 : 124 Sekunden, 7510 Einträge herausgenommen, bei 32562 Feldern (mit Wänden)
```

In [ ]:
# dein Code

<details>
<summary>Tipp anzeigen</summary>

Eine Schleife über `nummer` in `range(6)`, darin `lies_plan`, `finde`, `schnellste_route` und eine Ausgabe.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
for nummer in range(6):
    plan = lies_plan('daten/zauberschule' + str(nummer) + '.txt')
    sekunden, weg, herausgenommen = schnellste_route(plan, finde(plan, 'A'), finde(plan, 'B'))
    print('zauberschule' + str(nummer), ':', sekunden, 'Sekunden,', herausgenommen, 'Einträge herausgenommen, bei',
          2 * len(plan[0]) * len(plan[0][0]), 'Feldern (mit Wänden)')
```

Die Zeiten sind 8, 4, 14, 28, 84 und 124 Sekunden und stimmen mit dem Lösungsnotebook überein. Gegenüber dem oberen Stockwerk allein
(26, 4, 32, 28, 674, 1030) ist es bei 0, 2, 4 und 5 viel schneller, bei 1 und 3 lohnt sich der Wechsel nicht. Dijkstra hört
auf, sobald das Ziel an der Reihe ist, und sieht deshalb meist nicht alle Felder an.

</details>

#### Aufgabe E.5

Prüfe mit der Methode aus Abschnitt 5. Der Wechsel dauert 3 Sekunden: Statt einer Kante mit Kosten 3 baust du **zwei Zwischenfelder**.
Ein Zustand ist jetzt `(stock, zeile, spalte, phase)` mit `phase` 0 (normal), 1 und 2 (unterwegs). Aus Phase 0 geht
es zu Phase 1 (nur, wenn das Feld im anderen Stockwerk keine Wand ist), dann zu Phase 2 und dann zum Feld im anderen Stockwerk mit
Phase 0. Alle Kanten kosten 1. Die Breitensuche ist fertig, dir fehlt die Liste der Nachfolger.

**Erwartete Ausgabe:**
```
8
```

In [ ]:
from collections import deque

def zeit_mit_zwischenfeldern(plan, start, ziel):
    start = start + (0,)
    ziel = ziel + (0,)
    abstand = {start: 0}
    schlange = deque([start])
    while len(schlange) > 0 and ziel not in abstand:
        v = schlange.popleft()
        stock, r, c, phase = v
        nachfolger = []
        # dein Code: Liste der Nachfolger (jeweils mit vier Angaben)
        for u in nachfolger:
            if u not in abstand:
                abstand[u] = abstand[v] + 1
                schlange.append(u)
    return abstand.get(ziel)

print(zeit_mit_zwischenfeldern(plan0, start0, ziel0))

<details>
<summary>Tipp anzeigen</summary>

Phase 0: die Nachbarn mit Kosten 1 aus `nachbarn_schule` (mit `+ (0,)` zum Tupel mit vier Angaben) und, wenn der Wechsel
erlaubt ist, `(stock, r, c, 1)`. Phase 1: nur `(stock, r, c, 2)`. Phase 2: `(1 - stock, r, c, 0)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
from collections import deque

def zeit_mit_zwischenfeldern(plan, start, ziel):
    start = start + (0,)
    ziel = ziel + (0,)
    abstand = {start: 0}
    schlange = deque([start])
    while len(schlange) > 0 and ziel not in abstand:
        v = schlange.popleft()
        stock, r, c, phase = v
        nachfolger = []
        if phase == 0:
            for u, kosten in nachbarn_schule(plan, (stock, r, c)):
                if kosten == 1:
                    nachfolger.append(u + (0,))              # normaler Schritt
            if plan[1 - stock][r][c] != '#':
                nachfolger.append((stock, r, c, 1))          # Wechsel beginnt
        elif phase == 1:
            nachfolger.append((stock, r, c, 2))
        else:
            nachfolger.append((1 - stock, r, c, 0))          # angekommen im anderen Stockwerk
        for u in nachfolger:
            if u not in abstand:
                abstand[u] = abstand[v] + 1
                schlange.append(u)
    return abstand.get(ziel)

print(zeit_mit_zwischenfeldern(plan0, start0, ziel0))
```

Das Ergebnis ist dasselbe wie mit Dijkstra. Die Zwischenfelder sind dabei ganz normale Zustände.

</details>

Auch für alle sechs Beispiele muss dasselbe herauskommen. Das ist eine gute Probe, denn die beiden Verfahren sind unabhängig
voneinander geschrieben:

In [ ]:
for nummer in range(6):
    plan = lies_plan('daten/zauberschule' + str(nummer) + '.txt')
    a = finde(plan, 'A')
    b = finde(plan, 'B')
    dijkstra_zeit = schnellste_route(plan, a, b)[0]
    bfs_zeit = zeit_mit_zwischenfeldern(plan, a, b)
    print('zauberschule' + str(nummer), ':', dijkstra_zeit, bfs_zeit, dijkstra_zeit == bfs_zeit)

## Wettbewerbs-Ecke 2: Das ägyptische Grabmal (für Schnelle)

*Angelehnt an Aufgabe 5 „Das ägyptische Grabmal“ der 1. Runde des 43. Bundeswettbewerbs Informatik (2024/25).*

Petra steht am Eingang eines Gangs aus `n` Abschnitten, der zum Grabmal führt. In jedem Abschnitt hängt ein Steinquader mit einer
Periode `p` (in Minuten). Er **blockiert** den Abschnitt in den Minuten `0` bis `p − 1`, gibt ihn in den Minuten `p` bis `2p − 1`
frei, blockiert ihn wieder von `2p` bis `3p − 1` und so weiter. Ein Abschnitt ist also zur Minute `t` **offen**, wenn
`t % (2 * p) >= p` ist. Petra kann nur in offenen Abschnitten stehen. Zwischen zwei Abschnitten läuft sie ohne Zeitverlust, wenn
beide **in derselben Minute** offen sind. Wie früh kann sie im letzten Abschnitt sein?

Der Knoten ist ein **Zustand mit Zeit**: `(minute, ort)` mit `ort` 0 für den Eingang und 1 bis `n` für die Abschnitte. Die Kanten:
eine Minute **warten** (Kosten 1, der Ort muss dann noch offen sein) und in den **Nachbarabschnitt** laufen (Kosten 0, beide
offen). Die Kosten eines Zustands sind seine Minute. Dijkstra mit `(minute, ort)` in der Halde gibt die früheste Ankunft.
Das geht nur bei **kleinen** Minutenzahlen, denn es gibt `n · Minuten` Zustände. Das echte Beispiel `grabmal2` mit 490 021
Minuten hätte bis zu 2,4 Millionen Zustände, und `grabmal5` mit 85 Abschnitten und fast 44 Millionen Minuten gut 3,7 Milliarden.
Dafür gibt es das Lösungsnotebook, das statt der Minuten **Zeitfenster** betrachtet.

#### Aufgabe E.6

Schreibe `fruehestens(perioden)`: die Minute der frühesten Ankunft im letzten Abschnitt und die Zahl der herausgenommenen
Zustände. `lies_gang` und `ist_offen` sind fertig. Teste mit den Dateien `grabmal0`, `grabmal1` und `grabmal3`.

**Erwartete Ausgabe:**
```
grabmal0 : 3 Quader, (12, 24)
grabmal1 : 5 Quader, (49, 113)
grabmal3 : 10 Quader, (35, 119)
```

In [ ]:
def lies_gang(dateiname):
    zahlen = open(dateiname).read().split()
    return [int(x) for x in zahlen[1:1 + int(zahlen[0])]]

def ist_offen(p, minute):
    return minute % (2 * p) >= p

def fruehestens(perioden):
    n = len(perioden)
    gesehen = {(0, 0)}                       # Zustände (minute, ort), schon in der Halde
    warteschlange = [(0, 0)]
    herausgenommen = 0
    # dein Code
    return None, herausgenommen

for nummer in [0, 1, 3]:
    perioden = lies_gang('daten/grabmal' + str(nummer) + '.txt')
    print('grabmal' + str(nummer), ':', len(perioden), 'Quader,', fruehestens(perioden))

<details>
<summary>Tipp anzeigen</summary>

Aus `(minute, ort)` kommst du zu: `(minute + 1, ort)`, wenn der Ort der Eingang ist oder dann noch offen ist; `(minute, ort + 1)`,
wenn der Abschnitt `ort + 1` (Periode `perioden[ort]`) jetzt offen ist; `(minute, ort - 1)` ebenso, ab Abschnitt 2; vom Eingang
(Ort 0) nur in den ersten Abschnitt. Weil die Kosten eines Zustands seine Minute sind und die Halde nach der Minute ordnet,
darfst du einen Zustand schon beim Einreihen als gesehen merken.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lies_gang(dateiname):
    zahlen = open(dateiname).read().split()
    return [int(x) for x in zahlen[1:1 + int(zahlen[0])]]

def ist_offen(p, minute):
    return minute % (2 * p) >= p

def fruehestens(perioden):
    n = len(perioden)
    gesehen = {(0, 0)}                       # Zustände (minute, ort), schon in der Halde
    warteschlange = [(0, 0)]
    herausgenommen = 0
    while len(warteschlange) > 0:
        minute, ort = heappop(warteschlange)
        herausgenommen = herausgenommen + 1
        if ort == n:
            return minute, herausgenommen
        nachfolger = []
        if ort == 0 or ist_offen(perioden[ort - 1], minute + 1):
            nachfolger.append((minute + 1, ort))                 # eine Minute warten
        if ort == 0 and ist_offen(perioden[0], minute):
            nachfolger.append((minute, 1))                       # vom Eingang in den ersten Abschnitt
        if ort >= 1:
            if ort >= 2 and ist_offen(perioden[ort - 2], minute):
                nachfolger.append((minute, ort - 1))             # zurück
            if ort < n and ist_offen(perioden[ort], minute):
                nachfolger.append((minute, ort + 1))             # vor
        for z in nachfolger:
            if z not in gesehen:
                gesehen.add(z)
                heappush(warteschlange, z)
    return None, herausgenommen

for nummer in [0, 1, 3]:
    perioden = lies_gang('daten/grabmal' + str(nummer) + '.txt')
    print('grabmal' + str(nummer), ':', len(perioden), 'Quader,', fruehestens(perioden))
```

Die Zeiten 12, 49 und 35 stimmen mit den Lösungshinweisen überein. Dass man hier beim Einreihen markieren darf, ist kein
Widerspruch zu Fehler 4.1: Dort konnte ein Ort über verschiedene Wege verschieden früh erreicht werden. Hier gehört die Zeit
**zum Zustand**, ein Zustand `(minute, ort)` hat nur eine mögliche Kostenzahl.

</details>

## Das hast du gelernt

```python
from heapq import heappush, heappop

entfernung = {start: 0}
vorgaenger = {start: None}
warteschlange = [(0, start)]
while len(warteschlange) > 0:
    d, v = heappop(warteschlange)            # kleinste Entfernung zuerst
    if d > entfernung[v]:
        continue                             # veralteter Eintrag
    for u, kosten in nachbarn(v):
        neu = d + kosten
        if u not in entfernung or neu < entfernung[u]:
            entfernung[u] = neu
            vorgaenger[u] = v
            heappush(warteschlange, (neu, u))
```

- **Gewichtete Graphen:** Wörterbuch von Listen aus Paaren `(nachbar, gewicht)`. Die Breitensuche zählt nur Kanten und findet
  bei verschiedenen Gewichten nicht den kürzesten Weg.
- **`heapq`:** `heappush` und `heappop` mit Tupeln `(entfernung, knoten)`, das Kleinste kommt zuerst. Die Knoten müssen
  vergleichbar sein (alle Tupel, nicht Text und Tupel gemischt).
- **Endgültig** ist ein Knoten erst, wenn sein Eintrag **herauskommt**, nicht beim Einreihen. Veraltete Einträge überspringen.
  Das gilt nur bei Gewichten, die **nicht negativ** sind. Laufzeit O((V + E) · log V).
- **Weg:** Vorgänger mitführen und überschreiben, aufhören, wenn das Ziel herauskommt.
- **Zwischenknoten:** Bei kleinen ganzen Gewichten kann man Kanten zerlegen und die Breitensuche nehmen (Zauberschule: der
  Wechsel mit 3 Sekunden hat zwei Zwischenfelder).
- **Zustände mit Kosten:** (Station, Linie) mit Kosten für Fahrt und Umstieg, oder (Minute, Ort), wenn die Zeit im Zustand
  steckt.

### Aufgaben der 1. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Zauberschule](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/42_Zauberschule/zauberschule_loesung.ipynb) (42. BwInf): beide Stockwerke, Dijkstra und Zwischenfelder
- [Buhnenrennen](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/35_Buhnenrennen/buhnenrennen_loesung.ipynb) (35.),
  [Nummernmerker](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/38_Nummernmerker/nummernmerker_loesung.ipynb) (38.) und
  [Stadtführung](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/42_Stadtfuehrung/stadtfuehrung_loesung.ipynb) (42.): kürzeste Wege mit Gewichten
- [Das ägyptische Grabmal](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/43_AegyptischesGrabmal/aegyptischesgrabmal_loesung.ipynb) (43.): Zustand aus Ort und Zeit,
  mit Zeitfenstern statt einzelner Minuten

Im nächsten Notebook des Kerns, Notebook 10, geht es um **Merken** (Memoisierung): Was man schon berechnet hat, rechnet man
nicht noch einmal. Die Notebooks 08 und 09 kommen später.